In [1]:
include("CRD_STA.jl")
include("Fun.jl")
using NonlinearEigenproblems
using DelimitedFiles

In [11]:
N_cheb = 59
Mr = 0.3
gamma = 1.4
sigma = 0.72
Ro = 0.2
Co = 2 - Ro -Ro^2
Tw = 1
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,"cheb");
H,T = T_ca(Mr,f,q,w0,gamma,Tw);
F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim");
lam = - (2/3) * T;
kappa = (1/sigma) * T;

In [3]:
be=0.196
data_all = [0 0 0 0]
data = [0 0 0 0]
global R = 190
Ma = Mr/R
ali = 0.18
al = 0.35 + ali *im
alpha = []
B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
global C = eigen(B0,B1)
global val = filter(x->-0.03<imag(x)<0.03&&abs(real(x))<0.08,C.values)
while imag(val[1])>-0.001
    R = R-10
    Ma = Mr/R
    ali = 0.18
    al = 0.35 + ali *im
    alpha = []
    B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
    global C = eigen(B0,B1)
    global val = filter(x->-0.03<imag(x)<0.03&&abs(real(x))<0.08,C.values)
end
R

160

In [ ]:
data_total = [0 0 0 0 0 0]
for be = 0.1 : 0.002 : 0.3
    # be=0.173
    data_all = [0 0 0 0]
    data = [0 0 0 0]
    global R = 190
    Ma = Mr/R
    ali = 0.18
    al = 0.35 + ali *im
    alpha = []
    B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
    global C = eigen(B0,B1)
    global val = filter(x->-0.03<imag(x)<0.03&&abs(real(x))<0.08,C.values)
    while imag(val[1])>-0.001
        R = R-10
        Ma = Mr/R
        ali = 0.18
        al = 0.35 + ali *im
        alpha = []
        B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
        global C = eigen(B0,B1)
        global val = filter(x->-0.03<imag(x)<0.03&&abs(real(x))<0.05,C.values)
    end
    total = []
    Ma = Mr/R
    for ali = 0.18 : 0.002 : 0.4
        PinPoint= [0 0 0 0]
        al = 0.35 + ali *im
        alpha = []
        B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
        global C = eigen(B0,B1)
        global val = filter(x->-0.03<imag(x)<0.03&&abs(real(x))<0.05,C.values)

        for i = 1 : min(3,length(val))
            indi = []
            val_temp = val[i]
            for alr = 0.35 : 0.0015 : 0.5
                al = alr + ali *im
                vec = eigvector(val[i],C.values,C.vectors)
                B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
                val0,vec0 = RQI(B0,B1,val_temp,q0=vec)
                indi = [indi;val0]
                alpha = [alpha;al]
                val_temp = val0
                vec = vec0 
                end
            if i == 1
                total = indi
            else
                total = [total indi]
            end
        end
        for i = 1 : size(total,2)  

                d2 = diff1(real(total[:,i]),0.0015)
                
            for j = 1 : length(d2)-1

                if d2[j] * d2[j+1] < 0 && abs(d2[j+1])<0.005

                    PinPoint = [PinPoint;[R alpha[j,1] be total[j,i]]]

                end

            end
        end

        if PinPoint == [0 0 0 0]
            data_temp = [R be -1 -1]
        else
            data_temp = [R be PinPoint[findmax(imag(PinPoint[2:end,4]))[2] + 1,2] PinPoint[findmax(imag(PinPoint[2:end,4]))[2] + 1,4]]
        end
        data_all = [data_all;data_temp]
        writedlm("AS.dat",data_all[2:end,:])
        global data0 = data_all[2:end,:]
        if  length(axes(data0,1)) > 2 && imag(data0[end - 1,4])>imag(data0[end-2,4])&&imag(data0[end-1,4])>imag(data0[end,4])
            data = [data;data0[end-1:end-1,:]]
            writedlm("Dataall_$Tw _$Mr.dat",data0)
            break 
        elseif length(axes(data0,1)) > 2 && imag(data0[end - 2,4]) != -1 && imag(data0[end - 1,4]) == -1 && imag(data0[end,4]) == -1
            data = [data;data0[end-1:end-1,:]]
            writedlm("Dataall_$Tw _$Mr.dat",data[2:end,:])
            break 
        end
    end
    #section2
    global ali = imag(data[end,3])
    data_all = [0 0 0 0 0 0]
    data = [0 0 0 0]
    data1 = [0 0 0 0]
    for R = 155 : 0.5 : 220
        total = []
        Ma = Mr/R
        PinPoint= [0 0 0 0]
        al = 0.35 + ali *im
        alpha = []
        B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
        global C = eigen(B0,B1)
        global val = filter(x->-0.03<imag(x)<0.03&&abs(real(x))<0.1,C.values)

        for i = 1 : min(2,length(val))
            indi = []
            val_temp = val[i]
            for alr = 0.2 : 0.001 : 0.5
                al = alr + ali *im
                vec = eigvector(val[i],C.values,C.vectors)
                B0,B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co)
                val0,vec0 = RQI(B0,B1,val_temp,q0=vec)
                indi = [indi;val0]
                alpha = [alpha;al]
                val_temp = val0
                vec = vec0 
            end
            if i == 1
                total = indi
            else
                total = [total indi]
            end
        end
        for i = 1 : size(total,2)
            d2 = diff1(real(total[:,i]),0.001)
            for j = 1 : length(d2)-1
                if d2[j] * d2[j+1] < 0 && abs(d2[j+1])<0.005
                    PinPoint = [PinPoint;[R alpha[j,1] be total[j,i]]]
                end
            end
        end
        if PinPoint == [0 0 0 0]
            data_temp = [R be -1 -1]
        else
            data_temp = [R be PinPoint[findmax(imag(PinPoint[2:end,4]))[2] + 1,2] PinPoint[findmax(imag(PinPoint[2:end,4]))[2] + 1,4]]
        end
        data1 = [data1;data_temp]
        data_all = [real(data1[:,1]) real(data1[:,2]) real(data1[:,3]) imag(data1[:,3]) real(data1[:,4]) imag(data1[:,4])]
        writedlm("AS1.dat",data_all[2:end,:])
        if data_all[end,6]>0 && data_all[end-1,6]<0
            data_total = [data_total;data_all[end-1:end-1,:]]
            writedlm("Dataall_$Tw _$Mr.dat",data_total)
            break
        end
    end
end    

In [ ]:
ali

In [ ]:
plot!(data_all[2:end,1],data_all[2:end,6])

In [ ]:
data_all[end-1:end-1,:]

In [ ]:
PinPoint = [124 0.14 0.131+0.2im 0.3-0.3im;
            124 0.14 0.1+0.1im 0.25-0.1im;]
PinPoint[findmax(imag(PinPoint[:,4]))[2],3]
# PinPoint[2,3]
# findmax(imag(PinPoint[:,4]))[2]

In [ ]:
d2 = diff1(imag(total[:,1]),0.002)
@show PinPoint

In [ ]:
size(total,2)

In [ ]:
plot(d2,ylims = [-0.3,0.3])

In [ ]:
plot(real(total[:,1]),imag(total[:,1]))

In [ ]:
data = readdlm("Dataall_0.8 _0.3.dat")[3:end,:]
data_positive = (data[data[:,1] .== 20,:])
# c = data[:,1] .= 30
# norm(c)

In [ ]:
include("Fun.jl")

In [ ]:
dat = extline(0,"Dataall_0.8 _0.3.dat")

In [ ]:
scatter(dat[:,1],dat[:,4])